In [1]:
# ============================================================
# CELL 1 — IMPORTS AND PROJECT SETUP
# ============================================================

from __future__ import annotations

import gc
import inspect
import json
import math
import random
import re
import sys
import time

from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F


# ------------------------------------------------------------
# Locate project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "model").exists():

    for candidate in PROJECT_ROOT.parents:

        if (
            (candidate / "model").exists()
            and
            (candidate / "tokenizer").exists()
        ):
            PROJECT_ROOT = candidate
            break


if not (PROJECT_ROOT / "model").exists():

    # Final Windows fallback for this project
    fallback = Path(r"D:\Gpt2_v01")

    if fallback.exists():
        PROJECT_ROOT = fallback


if not (PROJECT_ROOT / "model").exists():

    raise RuntimeError(
        "Could not locate the MyGPT2 project root."
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ------------------------------------------------------------
# Project imports
# ------------------------------------------------------------

from model.config import GPTConfig
from model.model import MyGPTModel
from tokenizer.my_tokenizer import MyGPTTokenizer


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("=" * 80)
print("MYGPT2 CHECKPOINT EVALUATION")
print("=" * 80)

print()
print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")

if DEVICE.type == "cuda":

    print(
        f"GPU          : "
        f"{torch.cuda.get_device_name(0)}"
    )

print("=" * 80)

MYGPT2 CHECKPOINT EVALUATION

Project root : D:\Gpt2_v01
Device       : cuda
GPU          : NVIDIA GeForce RTX 5060 Ti


In [2]:
# ============================================================
# CELL 2 — EVALUATION CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

TOKENIZER_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tokenizer"
    / "tokenizer.json"
)


# ------------------------------------------------------------
# Optional manual checkpoint
# ------------------------------------------------------------
#
# Leave as None to automatically select the checkpoint with
# the highest known global step.
#
# Example:
#
# CHECKPOINT_PATH = (
#     PROJECT_ROOT
#     / "artifacts"
#     / "checkpoints"
#     / "latest.pt"
# )
#
# ------------------------------------------------------------

CHECKPOINT_PATH = None


# ------------------------------------------------------------
# Generation settings
# ------------------------------------------------------------

TEMPERATURE = 0.8
TOP_K = 50
TOP_P = 0.95

MAX_NEW_TOKENS = 100


# ------------------------------------------------------------
# Evaluation prompts
# ------------------------------------------------------------

PROMPTS = [

    "Once upon a time",

    "Artificial intelligence is changing the way humans",

    "Machine learning is a field of",

    "The little girl walked into the forest and",

    "In the future, computers may",

]


print("=" * 80)
print("EVALUATION CONFIGURATION")
print("=" * 80)

print()
print(f"Tokenizer : {TOKENIZER_PATH}")
print(f"Seed      : {SEED}")
print(f"Temp      : {TEMPERATURE}")
print(f"Top-k     : {TOP_K}")
print(f"Top-p     : {TOP_P}")

print("=" * 80)

EVALUATION CONFIGURATION

Tokenizer : D:\Gpt2_v01\artifacts\tokenizer\tokenizer.json
Seed      : 42
Temp      : 0.8
Top-k     : 50
Top-p     : 0.95


In [3]:
# ============================================================
# CELL 3 — LOCATE CHECKPOINT
# ============================================================

print("=" * 80)
print("LOCATING CHECKPOINT")
print("=" * 80)


CHECKPOINT_DIRECTORIES = [

    PROJECT_ROOT / "artifacts" / "checkpoints",

    PROJECT_ROOT / "checkpoints",

]


def extract_step_from_checkpoint(
    checkpoint_path: Path,
) -> int:
    """
    Determine checkpoint step without loading the full
    checkpoint whenever possible.
    """

    # --------------------------------------------------------
    # Prefer sidecar JSON metadata
    # --------------------------------------------------------

    metadata_path = checkpoint_path.with_suffix(".json")

    if metadata_path.exists():

        try:

            metadata = json.loads(
                metadata_path.read_text(
                    encoding="utf-8"
                )
            )

            step = metadata.get(
                "global_step"
            )

            if step is not None:
                return int(step)

        except Exception:
            pass


    # --------------------------------------------------------
    # Fall back to number in filename
    # --------------------------------------------------------

    numbers = re.findall(
        r"\d+",
        checkpoint_path.stem,
    )

    if numbers:

        try:
            return int(numbers[-1])

        except ValueError:
            pass


    # Unknown
    return -1


# ------------------------------------------------------------
# Manual path
# ------------------------------------------------------------

if CHECKPOINT_PATH is not None:

    CHECKPOINT_PATH = Path(
        CHECKPOINT_PATH
    ).resolve()

    if not CHECKPOINT_PATH.exists():

        raise FileNotFoundError(
            f"Checkpoint not found:\n"
            f"{CHECKPOINT_PATH}"
        )


# ------------------------------------------------------------
# Automatic selection
# ------------------------------------------------------------

else:

    checkpoint_candidates = []

    for directory in CHECKPOINT_DIRECTORIES:

        if not directory.exists():
            continue

        checkpoint_candidates.extend(
            directory.glob("*.pt")
        )

        checkpoint_candidates.extend(
            directory.glob("*.pth")
        )


    checkpoint_candidates = list(
        set(checkpoint_candidates)
    )


    if not checkpoint_candidates:

        raise FileNotFoundError(
            "No checkpoint files were found in:\n"
            + "\n".join(
                str(path)
                for path
                in CHECKPOINT_DIRECTORIES
            )
        )


    ranked_checkpoints = []

    for path in checkpoint_candidates:

        step = extract_step_from_checkpoint(
            path
        )

        ranked_checkpoints.append(
            (
                step,
                path.stat().st_mtime,
                path,
            )
        )


    ranked_checkpoints.sort(
        key=lambda item: (
            item[0],
            item[1],
        ),
        reverse=True,
    )


    CHECKPOINT_PATH = (
        ranked_checkpoints[0][2]
    )


print()
print(
    f"Selected checkpoint:\n"
    f"{CHECKPOINT_PATH}"
)

print()

print(
    f"Detected step from metadata/name : "
    f"{extract_step_from_checkpoint(CHECKPOINT_PATH):,}"
)

print(
    f"Checkpoint size                 : "
    f"{CHECKPOINT_PATH.stat().st_size / (1024 ** 2):.2f} MB"
)

print("=" * 80)

LOCATING CHECKPOINT

Selected checkpoint:
D:\Gpt2_v01\artifacts\checkpoints\final_step_00213751.pt

Detected step from metadata/name : 213,751
Checkpoint size                 : 1259.33 MB


In [4]:
# ============================================================
# CELL 4 — LOAD CHECKPOINT FILE
# ============================================================

print("=" * 80)
print("LOADING CHECKPOINT FILE")
print("=" * 80)


if not TOKENIZER_PATH.exists():

    raise FileNotFoundError(
        f"Tokenizer not found:\n"
        f"{TOKENIZER_PATH}"
    )


checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)


if not isinstance(
    checkpoint,
    dict,
):

    raise RuntimeError(
        "Checkpoint is not a dictionary."
    )


print()
print("Checkpoint keys:")

for key in checkpoint.keys():
    print(f"  {key}")


print()

print(
    f"Global step : "
    f"{checkpoint.get('global_step')}"
)

print(
    f"Epoch       : "
    f"{checkpoint.get('epoch')}"
)

print(
    f"Train loss  : "
    f"{checkpoint.get('train_loss')}"
)

print(
    f"Val loss    : "
    f"{checkpoint.get('val_loss')}"
)

print("=" * 80)

LOADING CHECKPOINT FILE

Checkpoint keys:
  checkpoint_version
  created_at
  epoch
  global_step
  best_loss
  train_loss
  val_loss
  model_state_dict
  optimizer_state_dict
  scheduler_state_dict
  random_states
  config
  extra
  mygpt2_training_manifest
  mygpt2_checkpoint_version
  mygpt2_checkpoint_saved_at

Global step : 213751
Epoch       : 0
Train loss  : 4.3478288650512695
Val loss    : None


In [5]:
# ============================================================
# CELL 5 — RESTORE MODEL CONFIGURATION
# ============================================================

print("=" * 80)
print("RESTORING MODEL CONFIGURATION")
print("=" * 80)


checkpoint_config = checkpoint.get(
    "config"
)


# ------------------------------------------------------------
# Start with project defaults
# ------------------------------------------------------------

config = GPTConfig()


# ------------------------------------------------------------
# Apply saved checkpoint configuration
# ------------------------------------------------------------

if isinstance(
    checkpoint_config,
    dict,
):

    for key, value in checkpoint_config.items():

        if hasattr(config, key):

            try:
                setattr(
                    config,
                    key,
                    value,
                )

            except Exception:

                print(
                    f"Could not restore config field: "
                    f"{key}"
                )


elif checkpoint_config is not None:

    # Handle the unlikely case where config itself was saved
    # as an object.

    for key in vars(config):

        if hasattr(
            checkpoint_config,
            key,
        ):

            try:

                setattr(
                    config,
                    key,
                    getattr(
                        checkpoint_config,
                        key,
                    ),
                )

            except Exception:
                pass


print()

CONFIG_FIELDS = [

    "vocab_size",
    "max_position_embeddings",
    "hidden_size",
    "num_layers",
    "num_attention_heads",
    "intermediate_size",
    "dropout",
    "attention_dropout",
    "embedding_dropout",
    "layer_norm_epsilon",
    "initializer_range",

]


for field in CONFIG_FIELDS:

    print(
        f"{field:<30}: "
        f"{getattr(config, field, 'N/A')}"
    )


print("=" * 80)

RESTORING MODEL CONFIGURATION

vocab_size                    : 32000
max_position_embeddings       : 512
hidden_size                   : 768
num_layers                    : 12
num_attention_heads           : 12
intermediate_size             : 3072
dropout                       : 0.1
attention_dropout             : 0.1
embedding_dropout             : 0.1
layer_norm_epsilon            : 1e-05
initializer_range             : 0.02


In [6]:
# ============================================================
# CELL 6 — LOCATE MODEL STATE
# ============================================================

print("=" * 80)
print("LOCATING MODEL STATE")
print("=" * 80)


MODEL_STATE_KEYS = [

    "model_state_dict",
    "model_state",
    "state_dict",
    "model",

]


model_state_dict = None
model_state_key = None


if isinstance(
    checkpoint,
    dict,
):

    for key in MODEL_STATE_KEYS:

        value = checkpoint.get(key)

        if isinstance(
            value,
            dict,
        ):

            tensor_values = [

                value_item

                for value_item
                in value.values()

                if torch.is_tensor(
                    value_item
                )

            ]

            if tensor_values:

                model_state_dict = value
                model_state_key = key

                break


# ------------------------------------------------------------
# Check if root itself is state dict
# ------------------------------------------------------------

if model_state_dict is None:

    tensor_values = [

        value

        for value
        in checkpoint.values()

        if torch.is_tensor(
            value
        )

    ]

    if tensor_values:

        model_state_dict = checkpoint
        model_state_key = "<root>"


if model_state_dict is None:

    raise RuntimeError(
        "Could not locate model state dictionary "
        "inside checkpoint."
    )


print()
print(
    f"Model state key : "
    f"{model_state_key}"
)

print(
    f"State tensors   : "
    f"{len(model_state_dict):,}"
)

print("=" * 80)

LOCATING MODEL STATE

Model state key : model_state_dict
State tensors   : 149


In [7]:
# ============================================================
# CELL 7 — LOAD TOKENIZER
# ============================================================

print("=" * 80)
print("LOADING TOKENIZER")
print("=" * 80)


tokenizer = MyGPTTokenizer.load(
    TOKENIZER_PATH
)


tokenizer_vocab_size = int(
    tokenizer.vocabulary_size
)


print()
print(
    f"Tokenizer vocabulary : "
    f"{tokenizer_vocab_size:,}"
)

print(
    f"Model vocabulary     : "
    f"{config.vocab_size:,}"
)


if (
    tokenizer_vocab_size
    !=
    config.vocab_size
):

    raise RuntimeError(
        "\nTOKENIZER / MODEL VOCABULARY MISMATCH\n\n"
        f"Tokenizer : {tokenizer_vocab_size}\n"
        f"Model     : {config.vocab_size}"
    )


print()
print(
    "PASS: tokenizer and model vocabulary match."
)

print("=" * 80)

LOADING TOKENIZER

Tokenizer vocabulary : 32,000
Model vocabulary     : 32,000

PASS: tokenizer and model vocabulary match.


In [8]:
# ============================================================
# CELL 8 — CONSTRUCT MODEL AND LOAD TRAINED WEIGHTS
# ============================================================

print("=" * 80)
print("LOADING TRAINED MODEL")
print("=" * 80)


# ------------------------------------------------------------
# Remove any old model from notebook memory
# ------------------------------------------------------------

if "model" in globals():

    try:
        del model
    except Exception:
        pass


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Construct architecture
# ------------------------------------------------------------

model = MyGPTModel(
    config
)


# ------------------------------------------------------------
# LOAD TRAINED WEIGHTS
#
# THIS IS THE CRITICAL STEP.
# ------------------------------------------------------------

load_result = model.load_state_dict(
    model_state_dict,
    strict=True,
)


# ------------------------------------------------------------
# Move trained model to device
# ------------------------------------------------------------

model = model.to(
    DEVICE
)


# ------------------------------------------------------------
# Evaluation mode
# ------------------------------------------------------------

model.eval()


print()

print(
    f"State source    : "
    f"{model_state_key}"
)

print(
    f"Missing keys    : "
    f"{len(load_result.missing_keys)}"
)

print(
    f"Unexpected keys : "
    f"{len(load_result.unexpected_keys)}"
)

print(
    f"Model device    : "
    f"{next(model.parameters()).device}"
)

print(
    f"Model dtype     : "
    f"{next(model.parameters()).dtype}"
)


if (
    len(load_result.missing_keys) != 0
    or
    len(load_result.unexpected_keys) != 0
):

    raise RuntimeError(
        "Checkpoint/model state mismatch."
    )


print()
print(
    "PASS: checkpoint weights loaded successfully."
)

print("=" * 80)

LOADING TRAINED MODEL

State source    : model_state_dict
Missing keys    : 0
Unexpected keys : 0
Model device    : cuda:0
Model dtype     : torch.float32

PASS: checkpoint weights loaded successfully.


In [9]:
# ============================================================
# CELL 9 — VERIFY CHECKPOINT WEIGHTS
# ============================================================

print("=" * 80)
print("CHECKPOINT WEIGHT VERIFICATION")
print("=" * 80)


loaded_state = model.state_dict()


checkpoint_keys = set(
    model_state_dict.keys()
)

model_keys = set(
    loaded_state.keys()
)


if checkpoint_keys != model_keys:

    raise RuntimeError(
        "Checkpoint/model tensor names do not match."
    )


matching_tensors = 0
different_tensors = 0

maximum_difference = 0.0


for name in model_state_dict:

    checkpoint_tensor = (
        model_state_dict[name]
        .detach()
        .cpu()
    )

    model_tensor = (
        loaded_state[name]
        .detach()
        .cpu()
    )


    if (
        checkpoint_tensor.shape
        !=
        model_tensor.shape
    ):

        raise RuntimeError(
            f"Shape mismatch for {name}"
        )


    difference = (
        checkpoint_tensor.float()
        -
        model_tensor.float()
    ).abs().max().item()


    maximum_difference = max(
        maximum_difference,
        difference,
    )


    if torch.equal(
        checkpoint_tensor,
        model_tensor,
    ):

        matching_tensors += 1

    else:

        different_tensors += 1


print()
print(
    f"Checkpoint tensors       : "
    f"{len(model_state_dict)}"
)

print(
    f"Exactly matching tensors : "
    f"{matching_tensors}"
)

print(
    f"Different tensors        : "
    f"{different_tensors}"
)

print(
    f"Maximum difference       : "
    f"{maximum_difference:.10f}"
)


if different_tensors != 0:

    raise RuntimeError(
        "FAIL: loaded model does not exactly match "
        "checkpoint weights."
    )


print()
print(
    "PASS: model is using the trained checkpoint weights."
)

print("=" * 80)

CHECKPOINT WEIGHT VERIFICATION

Checkpoint tensors       : 149
Exactly matching tensors : 149
Different tensors        : 0
Maximum difference       : 0.0000000000

PASS: model is using the trained checkpoint weights.


In [10]:
# ============================================================
# CELL 10 — MODEL SUMMARY
# ============================================================

print("=" * 80)
print("MODEL SUMMARY")
print("=" * 80)


total_parameters = sum(

    parameter.numel()

    for parameter
    in model.parameters()

)


trainable_parameters = sum(

    parameter.numel()

    for parameter
    in model.parameters()

    if parameter.requires_grad

)


print()

print(
    f"Vocabulary Size       : "
    f"{config.vocab_size:,}"
)

print(
    f"Context Length        : "
    f"{config.max_position_embeddings:,}"
)

print(
    f"Hidden Size           : "
    f"{config.hidden_size:,}"
)

print(
    f"Transformer Layers    : "
    f"{config.num_layers}"
)

print(
    f"Attention Heads       : "
    f"{config.num_attention_heads}"
)

print(
    f"Intermediate Size     : "
    f"{config.intermediate_size:,}"
)

print(
    f"Total Parameters      : "
    f"{total_parameters:,}"
)

print(
    f"Trainable Parameters  : "
    f"{trainable_parameters:,}"
)

print(
    f"Approx FP32 Size      : "
    f"{total_parameters * 4 / (1024 ** 2):.2f} MB"
)

print("=" * 80)

MODEL SUMMARY

Vocabulary Size       : 32,000
Context Length        : 512
Hidden Size           : 768
Transformer Layers    : 12
Attention Heads       : 12
Intermediate Size     : 3,072
Total Parameters      : 110,025,216
Trainable Parameters  : 110,025,216
Approx FP32 Size      : 419.71 MB


In [11]:
# ============================================================
# CELL 11 — FORWARD PASS SANITY CHECK
# ============================================================

print("=" * 80)
print("FORWARD PASS SANITY CHECK")
print("=" * 80)


test_text = (
    "Artificial intelligence can learn patterns from data."
)


test_ids = tokenizer.encode(
    test_text
)


if not test_ids:

    raise RuntimeError(
        "Tokenizer produced zero tokens."
    )


input_ids = torch.tensor(

    [test_ids],

    dtype=torch.long,

    device=DEVICE,

)


input_ids = input_ids[
    :,
    :config.max_position_embeddings
]


model.eval()


with torch.inference_mode():

    output = model(
        input_ids=input_ids
    )


if isinstance(
    output,
    tuple,
):

    logits = output[0]

elif hasattr(
    output,
    "logits",
):

    logits = output.logits

elif isinstance(
    output,
    dict,
):

    logits = output.get(
        "logits"
    )

elif torch.is_tensor(
    output
):

    logits = output

else:

    raise RuntimeError(
        "Could not extract logits from model output."
    )


if logits is None:

    raise RuntimeError(
        "Model returned no logits."
    )


expected_shape = (

    input_ids.shape[0],

    input_ids.shape[1],

    config.vocab_size,

)


print()
print(
    f"Input shape    : "
    f"{tuple(input_ids.shape)}"
)

print(
    f"Logits shape   : "
    f"{tuple(logits.shape)}"
)

print(
    f"Expected shape : "
    f"{expected_shape}"
)

print(
    f"Logits dtype   : "
    f"{logits.dtype}"
)


if (
    tuple(logits.shape)
    !=
    expected_shape
):

    raise RuntimeError(
        "Unexpected logits shape."
    )


print()
print(
    "PASS: forward pass completed successfully."
)

print("=" * 80)

FORWARD PASS SANITY CHECK

Input shape    : (1, 11)
Logits shape   : (1, 11, 32000)
Expected shape : (1, 11, 32000)
Logits dtype   : torch.float32

PASS: forward pass completed successfully.


In [12]:
# ============================================================
# CELL 12 — NUMERICAL STABILITY
# ============================================================

print("=" * 80)
print("NUMERICAL STABILITY CHECK")
print("=" * 80)


contains_nan = bool(
    torch.isnan(
        logits
    ).any().item()
)


contains_inf = bool(
    torch.isinf(
        logits
    ).any().item()
)


print()
print(
    f"Contains NaN : "
    f"{contains_nan}"
)

print(
    f"Contains Inf : "
    f"{contains_inf}"
)


if (
    contains_nan
    or
    contains_inf
):

    raise RuntimeError(
        "Invalid numerical output detected."
    )


print()
print(
    "PASS: numerical output is healthy."
)

print("=" * 80)

NUMERICAL STABILITY CHECK

Contains NaN : False
Contains Inf : False

PASS: numerical output is healthy.


In [13]:
# ============================================================
# CELL 13 — MANUAL LANGUAGE-MODEL LOSS TEST
# ============================================================

print("=" * 80)
print("MANUAL LANGUAGE-MODEL LOSS TEST")
print("=" * 80)


loss_test_text = (

    "Machine learning is a field of artificial intelligence "
    "that allows computer systems to learn patterns from data "
    "and use those patterns to make predictions."

)


token_ids = tokenizer.encode(
    loss_test_text
)


if len(token_ids) < 2:

    raise RuntimeError(
        "Loss-test text produced fewer than 2 tokens."
    )


tokens = torch.tensor(

    token_ids,

    dtype=torch.long,

    device=DEVICE,

)


tokens = tokens[
    :config.max_position_embeddings
]


input_ids = (
    tokens[:-1]
    .unsqueeze(0)
)


targets = (
    tokens[1:]
    .unsqueeze(0)
)


model.eval()


with torch.inference_mode():

    output = model(
        input_ids=input_ids
    )


if isinstance(
    output,
    tuple,
):

    manual_logits = output[0]

elif hasattr(
    output,
    "logits",
):

    manual_logits = output.logits

elif isinstance(
    output,
    dict,
):

    manual_logits = output["logits"]

else:

    manual_logits = output


manual_loss = F.cross_entropy(

    manual_logits.reshape(
        -1,
        manual_logits.size(-1),
    ),

    targets.reshape(-1),

)


manual_perplexity = math.exp(
    float(
        manual_loss.item()
    )
)


print()
print(
    f"Input tokens : "
    f"{input_ids.numel()}"
)

print(
    f"Loss         : "
    f"{manual_loss.item():.6f}"
)

print(
    f"Perplexity   : "
    f"{manual_perplexity:.4f}"
)

print("=" * 80)

MANUAL LANGUAGE-MODEL LOSS TEST

Input tokens : 26
Loss         : 4.155183
Perplexity   : 63.7636


In [14]:
# ============================================================
# CELL 14 — HELD-OUT LOSS / PERPLEXITY
# ============================================================

print("=" * 80)
print("HELD-OUT EVALUATION")
print("=" * 80)


evaluation_texts = [

    """
    Machine learning is a field of artificial intelligence
    that allows computer systems to learn patterns from data
    and use those patterns to make predictions.
    """,

    """
    The transformer architecture uses attention mechanisms
    to process relationships between tokens in a sequence.
    """,

    """
    A language model learns statistical relationships between
    tokens and can generate new text based on a sequence of
    previously observed tokens.
    """,

    """
    Python is a popular programming language used for machine
    learning, scientific computing, automation, and web
    development.
    """,

    """
    Neural networks contain layers of mathematical operations
    that transform input representations into useful outputs.
    """,

]


@torch.inference_mode()
def calculate_loss_and_perplexity(
    model,
    tokenizer,
    texts,
    sequence_length,
    device,
):

    model.eval()

    total_loss = 0.0
    total_predicted_tokens = 0


    for text in texts:

        ids = tokenizer.encode(
            text.strip()
        )


        if len(ids) < 2:
            continue


        ids = ids[
            :sequence_length
        ]


        token_tensor = torch.tensor(

            ids,

            dtype=torch.long,

            device=device,

        )


        inputs = (
            token_tensor[:-1]
            .unsqueeze(0)
        )


        targets = (
            token_tensor[1:]
            .unsqueeze(0)
        )


        output = model(
            input_ids=inputs
        )


        if isinstance(
            output,
            tuple,
        ):

            current_logits = output[0]

        elif hasattr(
            output,
            "logits",
        ):

            current_logits = output.logits

        elif isinstance(
            output,
            dict,
        ):

            current_logits = output[
                "logits"
            ]

        else:

            current_logits = output


        loss = F.cross_entropy(

            current_logits.reshape(
                -1,
                current_logits.size(-1),
            ),

            targets.reshape(-1),

            reduction="sum",

        )


        number_of_targets = (
            targets.numel()
        )


        total_loss += float(
            loss.item()
        )

        total_predicted_tokens += (
            number_of_targets
        )


    if total_predicted_tokens == 0:

        raise RuntimeError(
            "No usable tokens were found "
            "for held-out evaluation."
        )


    average_loss = (
        total_loss
        /
        total_predicted_tokens
    )


    perplexity = math.exp(
        average_loss
    )


    return (
        average_loss,
        perplexity,
        total_predicted_tokens,
    )


eval_loss, eval_perplexity, eval_token_count = (
    calculate_loss_and_perplexity(

        model=model,

        tokenizer=tokenizer,

        texts=evaluation_texts,

        sequence_length=(
            config.max_position_embeddings
        ),

        device=DEVICE,

    )
)


print()
print(
    f"Evaluated tokens       : "
    f"{eval_token_count:,}"
)

print(
    f"Evaluation Loss        : "
    f"{eval_loss:.6f}"
)

print(
    f"Evaluation Perplexity  : "
    f"{eval_perplexity:.4f}"
)

print("=" * 80)

HELD-OUT EVALUATION

Evaluated tokens       : 128
Evaluation Loss        : 7.090229
Evaluation Perplexity  : 1200.1825


In [15]:
# ============================================================
# CELL 15 — CHECKPOINT LOSS COMPARISON
# ============================================================

print("=" * 80)
print("CHECKPOINT LOSS COMPARISON")
print("=" * 80)


saved_train_loss = checkpoint.get(
    "train_loss"
)


saved_val_loss = checkpoint.get(
    "val_loss"
)


print()

if (
    saved_train_loss is not None
    and
    math.isfinite(
        float(saved_train_loss)
    )
):

    saved_train_loss = float(
        saved_train_loss
    )

    saved_train_perplexity = math.exp(
        saved_train_loss
    )

    print(
        f"Saved Train Loss       : "
        f"{saved_train_loss:.6f}"
    )

    print(
        f"Saved Train Perplexity : "
        f"{saved_train_perplexity:.4f}"
    )

else:

    saved_train_perplexity = None

    print(
        "Saved Train Loss       : unavailable"
    )


print()

print(
    f"Held-Out Loss          : "
    f"{eval_loss:.6f}"
)

print(
    f"Held-Out Perplexity    : "
    f"{eval_perplexity:.4f}"
)


if (
    saved_val_loss is not None
    and
    math.isfinite(
        float(saved_val_loss)
    )
):

    print()

    print(
        f"Saved Validation Loss  : "
        f"{float(saved_val_loss):.6f}"
    )


print("=" * 80)

CHECKPOINT LOSS COMPARISON

Saved Train Loss       : 4.347829
Saved Train Perplexity : 77.3104

Held-Out Loss          : 7.090229
Held-Out Perplexity    : 1200.1825


In [16]:
# ============================================================
# CELL 16 — SAMPLING HELPER
# ============================================================

def sample_next_token(
    logits: torch.Tensor,
    *,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.95,
) -> int:


    if temperature <= 0:

        raise ValueError(
            "temperature must be > 0"
        )


    if not (
        0 < top_p <= 1
    ):

        raise ValueError(
            "top_p must be in (0, 1]"
        )


    logits = (
        logits.float()
        /
        temperature
    )


    # --------------------------------------------------------
    # Top-k filtering
    # --------------------------------------------------------

    if top_k > 0:

        k = min(
            int(top_k),
            logits.shape[-1],
        )


        threshold = torch.topk(
            logits,
            k,
        ).values[-1]


        logits = logits.masked_fill(
            logits < threshold,
            float("-inf"),
        )


    # --------------------------------------------------------
    # Top-p / nucleus filtering
    # --------------------------------------------------------

    if top_p < 1.0:

        sorted_logits, sorted_indices = (
            torch.sort(
                logits,
                descending=True,
            )
        )


        sorted_probabilities = (
            torch.softmax(
                sorted_logits,
                dim=-1,
            )
        )


        cumulative_probabilities = (
            torch.cumsum(
                sorted_probabilities,
                dim=-1,
            )
        )


        remove_mask = (
            cumulative_probabilities
            >
            top_p
        )


        remove_mask[1:] = (
            remove_mask[:-1].clone()
        )


        remove_mask[0] = False


        sorted_logits = (
            sorted_logits.masked_fill(
                remove_mask,
                float("-inf"),
            )
        )


        filtered_logits = (
            torch.full_like(
                logits,
                float("-inf"),
            )
        )


        filtered_logits.scatter_(
            0,
            sorted_indices,
            sorted_logits,
        )


        logits = filtered_logits


    probabilities = torch.softmax(
        logits,
        dim=-1,
    )


    if not torch.isfinite(
        probabilities
    ).all():

        raise RuntimeError(
            "Invalid sampling probabilities."
        )


    token = torch.multinomial(
        probabilities,
        num_samples=1,
    )


    return int(
        token.item()
    )


print(
    "Sampling helper ready."
)

Sampling helper ready.


In [17]:
# ============================================================
# CELL 17 — TEXT GENERATION FUNCTION
# ============================================================

@torch.inference_mode()
def generate_text(
    prompt: str,
    *,
    max_new_tokens: int = 100,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.95,
):

    model.eval()


    prompt_ids = tokenizer.encode(
        prompt
    )


    if not prompt_ids:

        raise ValueError(
            "Prompt produced zero tokens."
        )


    original_prompt_length = len(
        prompt_ids
    )


    generated_ids = list(
        prompt_ids
    )


    context_limit = int(
        config.max_position_embeddings
    )


    eos_token_id = getattr(
        config,
        "eos_token_id",
        None,
    )


    for _ in range(
        max_new_tokens
    ):


        # Model input is limited to context window,
        # but generated_ids keeps complete output.

        context_ids = generated_ids[
            -context_limit:
        ]


        input_ids = torch.tensor(

            [context_ids],

            dtype=torch.long,

            device=DEVICE,

        )


        output = model(
            input_ids=input_ids
        )


        if isinstance(
            output,
            tuple,
        ):

            generation_logits = output[0]

        elif hasattr(
            output,
            "logits",
        ):

            generation_logits = (
                output.logits
            )

        elif isinstance(
            output,
            dict,
        ):

            generation_logits = output[
                "logits"
            ]

        else:

            generation_logits = output


        next_token_id = sample_next_token(

            generation_logits[
                0,
                -1,
                :
            ],

            temperature=temperature,

            top_k=top_k,

            top_p=top_p,

        )


        generated_ids.append(
            next_token_id
        )


        if (
            eos_token_id is not None
            and
            next_token_id
            ==
            eos_token_id
        ):

            break


    generated_text = tokenizer.decode(
        generated_ids
    )


    actual_new_tokens = (
        len(generated_ids)
        -
        original_prompt_length
    )


    return {

        "text": generated_text,

        "ids": generated_ids,

        "prompt_tokens": (
            original_prompt_length
        ),

        "generated_tokens": (
            actual_new_tokens
        ),

    }


print(
    "Text generation function ready."
)

Text generation function ready.


In [18]:
# ============================================================
# CELL 18 — GENERATION QUALITY TEST
# ============================================================

print("=" * 80)
print("TEXT GENERATION QUALITY TEST")
print("=" * 80)


generation_results = []


for index, prompt in enumerate(
    PROMPTS,
    start=1,
):

    result = generate_text(

        prompt,

        max_new_tokens=MAX_NEW_TOKENS,

        temperature=TEMPERATURE,

        top_k=TOP_K,

        top_p=TOP_P,

    )


    generation_results.append(
        result
    )


    print()
    print("=" * 80)

    print(
        f"PROMPT {index}"
    )

    print("-" * 80)

    print(
        f"Prompt: {prompt}"
    )

    print()

    print(
        result["text"]
    )

    print()

    print(
        f"Generated tokens: "
        f"{result['generated_tokens']}"
    )


print()
print("=" * 80)

TEXT GENERATION QUALITY TEST

PROMPT 1
--------------------------------------------------------------------------------
Prompt: Once upon a time

 Once upon a time was the real thing: the first time the land was full of trees, animals, trees and trees. One of the first things we noticed was that the trees are not very big enough for a lot of people. What we noticed was that the trees were very big enough for an even bigger than the trees. It looked a lot like a garden with a lot of trees. Then came the tree.

We noticed that one of the first trees we noticed was that they had many trees. We

Generated tokens: 100

PROMPT 2
--------------------------------------------------------------------------------
Prompt: Artificial intelligence is changing the way humans

 Artificial intelligence is changing the way humans up, and a new study by researchers from the University of Texas, San Francisco, shows that human activity in the brain has been linked to an increasing number of brain activity

In [19]:
# ============================================================
# CELL 19 — GENERATION EFFICIENCY
# ============================================================

print("=" * 80)
print("TEXT GENERATION EFFICIENCY TEST")
print("=" * 80)


benchmark_prompt = (
    "Artificial intelligence is changing the way humans"
)


BENCHMARK_NEW_TOKENS = 100


# ------------------------------------------------------------
# Reset GPU statistics
# ------------------------------------------------------------

if DEVICE.type == "cuda":

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats(
        DEVICE
    )

    torch.cuda.synchronize(
        DEVICE
    )


# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

start_time = time.perf_counter()


benchmark_result = generate_text(

    benchmark_prompt,

    max_new_tokens=BENCHMARK_NEW_TOKENS,

    temperature=TEMPERATURE,

    top_k=TOP_K,

    top_p=TOP_P,

)


if DEVICE.type == "cuda":

    torch.cuda.synchronize(
        DEVICE
    )


end_time = time.perf_counter()


elapsed_time = (
    end_time
    -
    start_time
)


generated_tokens = (
    benchmark_result[
        "generated_tokens"
    ]
)


tokens_per_second = (

    generated_tokens
    /
    elapsed_time

    if elapsed_time > 0

    else 0.0

)


milliseconds_per_token = (

    elapsed_time
    /
    generated_tokens
    *
    1000

    if generated_tokens > 0

    else 0.0

)


if DEVICE.type == "cuda":

    peak_gpu_memory_mb = (

        torch.cuda.max_memory_allocated(
            DEVICE
        )
        /
        (1024 ** 2)

    )

else:

    peak_gpu_memory_mb = 0.0


print()

print(
    f"Prompt                : "
    f"{benchmark_prompt}"
)

print(
    f"Prompt tokens         : "
    f"{benchmark_result['prompt_tokens']}"
)

print(
    f"Generated tokens      : "
    f"{generated_tokens}"
)

print()

print("-" * 80)

print(
    benchmark_result["text"]
)

print("-" * 80)

print()

print(
    f"Generation time       : "
    f"{elapsed_time:.4f} sec"
)

print(
    f"Tokens / second       : "
    f"{tokens_per_second:.2f}"
)

print(
    f"Milliseconds / token  : "
    f"{milliseconds_per_token:.2f} ms"
)


if DEVICE.type == "cuda":

    print(
        f"Peak GPU memory       : "
        f"{peak_gpu_memory_mb:.2f} MB"
    )


print("=" * 80)

TEXT GENERATION EFFICIENCY TEST

Prompt                : Artificial intelligence is changing the way humans
Prompt tokens         : 10
Generated tokens      : 100

--------------------------------------------------------------------------------
 Artificial intelligence is changing the way humans

A new study published last week by the University of California at California University’s Berkeley Medical School found that the brain can produce a significant amount of brain damage by providing an even faster-paced and more reliable means for people with a more reliable sense of the world.

The study was published in the journal Nature Science & Technology (CJJJ) in a paper published last year in the journal Nature Science & Technology. The new study, published in the journal Nature and Science
--------------------------------------------------------------------------------

Generation time       : 0.6278 sec
Tokens / second       : 159.29
Milliseconds / token  : 6.28 ms
Peak GPU memory   

In [20]:
# ============================================================
# CELL 20 — GENERATION SCALING BENCHMARK
# ============================================================

print("=" * 80)
print("GENERATION SCALING BENCHMARK")
print("=" * 80)


BENCHMARK_LENGTHS = [

    20,
    50,
    100,
    200,

]


speed_results = []


for token_target in BENCHMARK_LENGTHS:


    if DEVICE.type == "cuda":

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats(
            DEVICE
        )

        torch.cuda.synchronize(
            DEVICE
        )


    start = time.perf_counter()


    result = generate_text(

        benchmark_prompt,

        max_new_tokens=token_target,

        temperature=TEMPERATURE,

        top_k=TOP_K,

        top_p=TOP_P,

    )


    if DEVICE.type == "cuda":

        torch.cuda.synchronize(
            DEVICE
        )


    elapsed = (
        time.perf_counter()
        -
        start
    )


    actual_tokens = (
        result[
            "generated_tokens"
        ]
    )


    rate = (

        actual_tokens
        /
        elapsed

        if elapsed > 0

        else 0.0

    )


    if DEVICE.type == "cuda":

        memory_mb = (

            torch.cuda.max_memory_allocated(
                DEVICE
            )
            /
            (1024 ** 2)

        )

    else:

        memory_mb = 0.0


    speed_results.append({

        "requested_tokens": token_target,

        "generated_tokens": actual_tokens,

        "time_seconds": elapsed,

        "tokens_per_second": rate,

        "peak_memory_mb": memory_mb,

    })


print()

print(
    f"{'Tokens':>10} "
    f"{'Time(s)':>12} "
    f"{'Tokens/s':>12} "
    f"{'Peak MB':>12}"
)

print("-" * 50)


for result in speed_results:

    print(

        f"{result['generated_tokens']:>10} "

        f"{result['time_seconds']:>12.4f} "

        f"{result['tokens_per_second']:>12.2f} "

        f"{result['peak_memory_mb']:>12.2f}"

    )


print("=" * 80)

GENERATION SCALING BENCHMARK

    Tokens      Time(s)     Tokens/s      Peak MB
--------------------------------------------------
        20       0.1191       167.90       464.68
        50       0.2826       176.94       472.09
       100       0.6287       159.06       484.45
       200       1.4765       135.45       510.25


In [21]:
# ============================================================
# CELL 21 — TEMPERATURE COMPARISON
# ============================================================

print("=" * 80)
print("TEMPERATURE COMPARISON")
print("=" * 80)


comparison_prompt = (
    "Once upon a time"
)


for temperature in [

    0.6,
    0.8,
    1.0,

]:

    # Same seed makes comparison more meaningful

    random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            SEED
        )


    result = generate_text(

        comparison_prompt,

        max_new_tokens=80,

        temperature=temperature,

        top_k=TOP_K,

        top_p=TOP_P,

    )


    print()

    print("=" * 80)

    print(
        f"TEMPERATURE = "
        f"{temperature}"
    )

    print("-" * 80)

    print(
        result["text"]
    )


print()
print("=" * 80)

TEMPERATURE COMPARISON

TEMPERATURE = 0.6
--------------------------------------------------------------------------------
 Once upon a time was the real thing.

The big question was, what would you say?

I think you could say it was a long time ago, but it was probably a long time ago. It was just a long time ago, and I was thinking, what would you say?

I thought it was a long time ago, and I was thinking, what would you say?


TEMPERATURE = 0.8
--------------------------------------------------------------------------------
 Once upon a time was the real thing: the first time the land was full of trees, animals, trees and trees. One of the first things we noticed was that the trees are not very big enough for a lot of people. What we noticed was that the trees were very big enough for an even bigger than the trees. It looked a lot like a garden with a lot of trees. Then came the tree.

TEMPERATURE = 1.0
--------------------------------------------------------------------------------

In [22]:
# ============================================================
# CELL 22 — OPTIONAL TRAINING LOG ANALYSIS
# ============================================================

print("=" * 80)
print("TRAINING LOG ANALYSIS")
print("=" * 80)


LOG_DIRECTORY = (
    PROJECT_ROOT
    /
    "logs"
)


POSSIBLE_LOGS = [

    LOG_DIRECTORY / "training.log",

    LOG_DIRECTORY / "train.log",

    PROJECT_ROOT / "training.log",

    PROJECT_ROOT / "train.log",

]


steps = []
losses = []
learning_rates = []

LOG_PATH = None


for path in POSSIBLE_LOGS:

    if path.exists():

        LOG_PATH = path
        break


if LOG_PATH is None:

    print()

    print(
        "No dedicated training log found."
    )

    print(
        "Training-history analysis will "
        "be skipped safely."
    )


else:

    print()
    print(
        f"Training log: "
        f"{LOG_PATH}"
    )


    log_text = (
        LOG_PATH.read_text(

            encoding="utf-8",

            errors="ignore",

        )
    )


    pattern = re.compile(

        r"Step\s+(\d+)\s*\|\s*"
        r"Loss\s+([0-9.eE+-]+)\s*\|\s*"
        r"LR\s+([0-9.eE+-]+)"

    )


    matches = pattern.findall(
        log_text
    )


    for step, loss, lr in matches:

        steps.append(
            int(step)
        )

        losses.append(
            float(loss)
        )

        learning_rates.append(
            float(lr)
        )


    print()

    print(
        f"Logged steps : "
        f"{len(steps):,}"
    )


    if steps:

        print(
            f"First step   : "
            f"{steps[0]:,}"
        )

        print(
            f"Last step    : "
            f"{steps[-1]:,}"
        )

        print(
            f"First loss   : "
            f"{losses[0]:.6f}"
        )

        print(
            f"Last loss    : "
            f"{losses[-1]:.6f}"
        )


print("=" * 80)

TRAINING LOG ANALYSIS

No dedicated training log found.
Training-history analysis will be skipped safely.


In [23]:
# ============================================================
# CELL 23 — OPTIONAL TRAINING CURVES
# ============================================================

if steps:

    plt.figure(
        figsize=(12, 5)
    )

    plt.plot(
        steps,
        losses,
    )

    plt.xlabel(
        "Training Step"
    )

    plt.ylabel(
        "Loss"
    )

    plt.title(
        "Training Loss"
    )

    plt.grid(
        alpha=0.3
    )

    plt.show()


    plt.figure(
        figsize=(12, 5)
    )

    plt.plot(
        steps,
        learning_rates,
    )

    plt.xlabel(
        "Training Step"
    )

    plt.ylabel(
        "Learning Rate"
    )

    plt.title(
        "Learning Rate Schedule"
    )

    plt.grid(
        alpha=0.3
    )

    plt.show()


else:

    print(
        "Training log unavailable — "
        "training plots skipped."
    )

Training log unavailable — training plots skipped.


In [24]:
# ============================================================
# CELL 24 — FINAL EVALUATION SUMMARY
# ============================================================

print("=" * 80)
print("MYGPT2 CHECKPOINT EVALUATION SUMMARY")
print("=" * 80)

print()

print(
    f"Checkpoint File        : "
    f"{CHECKPOINT_PATH.name}"
)

print(
    f"Checkpoint Step        : "
    f"{checkpoint.get('global_step')}"
)

print(
    f"Checkpoint Epoch       : "
    f"{checkpoint.get('epoch')}"
)

print()

print(
    f"Vocabulary Size        : "
    f"{config.vocab_size:,}"
)

print(
    f"Context Length         : "
    f"{config.max_position_embeddings:,}"
)

print(
    f"Hidden Size            : "
    f"{config.hidden_size:,}"
)

print(
    f"Transformer Layers     : "
    f"{config.num_layers}"
)

print(
    f"Attention Heads        : "
    f"{config.num_attention_heads}"
)

print(
    f"Intermediate Size      : "
    f"{config.intermediate_size:,}"
)

print(
    f"Total Parameters       : "
    f"{total_parameters:,}"
)

print()

print(
    f"Model Device           : "
    f"{next(model.parameters()).device}"
)

print(
    f"Model State Tensors    : "
    f"{len(model_state_dict)}"
)

print(
    f"Matching Tensors       : "
    f"{matching_tensors}"
)

print(
    f"Different Tensors      : "
    f"{different_tensors}"
)

print()

if saved_train_loss is not None:

    print(
        f"Saved Training Loss    : "
        f"{saved_train_loss:.6f}"
    )

    print(
        f"Saved Training PPL     : "
        f"{saved_train_perplexity:.4f}"
    )


print(
    f"Manual Test Loss       : "
    f"{manual_loss.item():.6f}"
)

print(
    f"Manual Test PPL        : "
    f"{manual_perplexity:.4f}"
)

print(
    f"Held-Out Loss          : "
    f"{eval_loss:.6f}"
)

print(
    f"Held-Out Perplexity    : "
    f"{eval_perplexity:.4f}"
)

print()

print(
    f"Generation Speed       : "
    f"{tokens_per_second:.2f} tokens/sec"
)

print(
    f"Generation Latency     : "
    f"{milliseconds_per_token:.2f} ms/token"
)


if DEVICE.type == "cuda":

    print(
        f"Peak GPU Memory        : "
        f"{peak_gpu_memory_mb:.2f} MB"
    )


print()

if LOG_PATH is None:

    print(
        "Training Log           : unavailable "
        "(skipped safely)"
    )

else:

    print(
        f"Training Log           : "
        f"{LOG_PATH.name}"
    )


print()
print("-" * 80)

print(
    "Checkpoint loading     : PASS"
)

print(
    "Weight verification    : PASS"
)

print(
    "Forward pass           : PASS"
)

print(
    "Numerical stability    : PASS"
)

print(
    "Loss evaluation        : COMPLETE"
)

print(
    "Text generation        : COMPLETE"
)

print(
    "Efficiency benchmark   : COMPLETE"
)

print("-" * 80)

print()
print(
    "CHECKPOINT EVALUATION COMPLETE"
)

print("=" * 80)

MYGPT2 CHECKPOINT EVALUATION SUMMARY

Checkpoint File        : final_step_00213751.pt
Checkpoint Step        : 213751
Checkpoint Epoch       : 0

Vocabulary Size        : 32,000
Context Length         : 512
Hidden Size            : 768
Transformer Layers     : 12
Attention Heads        : 12
Intermediate Size      : 3,072
Total Parameters       : 110,025,216

Model Device           : cuda:0
Model State Tensors    : 149
Matching Tensors       : 149
Different Tensors      : 0

Saved Training Loss    : 4.347829
Saved Training PPL     : 77.3104
Manual Test Loss       : 4.155183
Manual Test PPL        : 63.7636
Held-Out Loss          : 7.090229
Held-Out Perplexity    : 1200.1825

Generation Speed       : 159.29 tokens/sec
Generation Latency     : 6.28 ms/token
Peak GPU Memory        : 484.45 MB

Training Log           : unavailable (skipped safely)

--------------------------------------------------------------------------------
Checkpoint loading     : PASS
Weight verification    : PASS
For